In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.types import *

c_schema = StructType(
    [
        StructField('customer_id',StringType()),
        StructField('customer_name',StringType()),
        StructField('customer_email',StringType()),
        StructField('customer_city',StringType()),
        StructField('segment',StringType()),
        StructField('customer_phone',StringType()),
        StructField('load_dt',StringType())
    ]
)

o_schema = StructType(
    [
        StructField('order_id',IntegerType()),
        StructField('customer_id',StringType()),
        StructField('product_id',IntegerType()),
        StructField('amount',IntegerType()),
        StructField('order_dt',StringType())
    ]
)

p_schema = StructType(
    [
        StructField('product_id',FloatType()),
        StructField('product_name',StringType()),
        StructField('category',StringType()),
        StructField('brand',StringType()),
        StructField('price',IntegerType()),
        StructField('active_flag',StringType())
    ]
)

b_customers = spark.read.format('csv').option('header','true').schema(c_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoG_V2/customers_source.csv')
b_orders = spark.read.format('csv').option('header','true').schema(o_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoG_V2/orders_source.csv')
b_products = spark.read.format('csv').option('header','true').schema(p_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoG_V2/products_source.csv')

# Clean products
b_products = b_products.dropna(subset=('product_id'))
b_products = b_products.dropDuplicates()

# Audit Columns
b_products = b_products.withColumn('products_create_dt',f.lit(f.current_date())).withColumn('products_update_dt',f.lit(None))

b_products = b_products.fillna({
    'product_name': 'Unknown',
    'category': 'Unknown',
    'brand': 'Unknown',
    'price':0,
    'active_flag': 'Unknown'
})

b_products = b_products.withColumn('product_name',f.initcap(f.trim(f.col('product_name')))).withColumn('category',f.initcap(f.trim(f.col('category')))).withColumn('brand',f.initcap(f.trim(f.col('brand')))).withColumn('active_flag',f.initcap(f.trim(f.col('active_flag'))))

b_products = b_products.filter((f.col('price')>=0))

# Clean orders

b_orders = b_orders.dropna(subset=('order_id'))
b_orders = b_orders.dropDuplicates()
# b_orders = b_orders.withColumn('order_dt',f.col('order_dt').cast(DateType()))
b_orders = b_orders.withColumn('customer_id',f.col('customer_id').cast(FloatType()))
b_orders = b_orders.withColumn('customer_id',f.col('customer_id').cast(IntegerType()))

# Audit Columns 
b_orders = b_orders.withColumn('orders_create_dt',f.lit(f.current_date())).withColumn('orders_updated_dt',f.lit(None))

# Standardizr data
b_orders = b_orders.fillna({
    'customer_id':0,
    'product_id':0,
    'amount':0
})

#(f.col('order_dt') <= f.current_date()
b_orders = b_orders.filter((f.col('amount') >= 0))
b_orders = b_orders.withColumnRenamed('amount','orders_amount')



# Clean customers 
b_customers = b_customers.withColumn('customer_load_dt',f.col('load_dt').cast(DateType())).withColumn('customer_id',f.col('customer_id').cast(FloatType()))
b_customers = b_customers.dropna(subset=('customer_id'))
b_customers = b_customers.dropDuplicates()
w = Window.partitionBy('customer_id').orderBy(f.col('load_dt').desc())
b_customers = b_customers.withColumn('rn',f.row_number().over(w)).filter(f.col('rn')==1).drop('rn')
b_customers = b_customers.drop('load_dt')

# Add Audit Column
b_customers = b_customers.withColumn('customer_created_dt',f.lit(f.current_date())).withColumn('customer_updated_dt',f.lit(None))

# Standardize data
b_customers = b_customers.fillna({
    'customer_name':'Unknown',
    'customer_email':'Unknown',
    'customer_city':'Unknown',
    'segment':'Unknown',
    'customer_phone':'Unknown',
})

phone_regex = r'[^0-9]'
b_customers = b_customers.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('customer_email',f.lower(f.trim(f.col('customer_email')))).withColumn('customer_city',f.initcap(f.trim(f.col('customer_city')))).withColumn('segment',f.initcap(f.trim(f.col('segment')))).withColumn('customer_phone',f.regexp_replace(f.col('customer_phone'),phone_regex,''))

# Data Validation
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

phone_validation = (f.length(f.col('customer_phone'))==10)
email_validation = (f.col('customer_email').rlike(email_regex))

b_customers = b_customers.withColumn('customer_id',f.col('customer_id').cast(IntegerType()))

b_customers = b_customers.withColumn('data_quality',
                                     f.when(
                                         (email_validation) & (phone_validation) & 
                                            (f.col('customer_load_dt')<=f.current_date()
                                        ),'pass').otherwise('fail'))


silver_customers = b_customers.filter(f.col('data_quality')=='pass')
error_customers = b_customers.filter(f.col('data_quality')=='fail')

# silver_customers.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_customers')

# b_orders = b_orders.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_orders')

# b_products = b_products.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_products')

# Lookup table
gold_orders = (b_orders.alias('o').join(silver_customers.alias('c'),on='customer_id',how='left').join(b_products.alias('p'),on='product_id',how='left')).select('c.customer_id','c.customer_name','c.customer_email','c.customer_city','c.segment','c.customer_phone','p.product_id','p.product_name','p.category','p.brand','p.price','p.active_flag','p.products_create_dt','p.products_update_dt','o.order_id','orders_amount','o.order_dt','orders_create_dt','orders_updated_dt')

gold_orders.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('gold_orders')

# Merge Operations

c_incremental = spark.read.format('csv').schema(c_schema).option('header','true').load('/Workspace/Users/manikantak177@gmail.com/BtoStoG_V2/customers_incremental.csv')

# # Load_dt
# c_incremental = c_incremental.withColumn('load_dt',f.col('load_dt').cast(DateType()))

# Drop duplicates
c_incremental = c_incremental.dropna(subset=('customer_id'))

# Drop Duplicates 
c_incremental = c_incremental.dropDuplicates()
w = Window.partitionBy('customer_id').orderBy(f.col('load_dt').desc())
c_incremental = c_incremental.withColumn('rn',f.row_number().over(w)).filter(f.col('rn')==1).drop('rn')

# Fill null
c_incremental = c_incremental.fillna({
    'customer_name':'Unknown',
    'customer_email':'Unknown',
    'customer_city':'Unknown',
    'segment':'Unknown',
    'customer_phone':'0'
})

phone_regex = r'[^0-9]'
# Standaizaton data
c_incremental = c_incremental.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('customer_email',f.initcap(f.trim(f.col('customer_email')))).withColumn('customer_city',f.initcap(f.trim('customer_city'))).withColumn('segment',f.initcap(f.trim(f.col('segment')))).withColumn('customer_phone',f.regexp_replace(f.col('customer_phone'),phone_regex,''))

# Data Validation
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

email_validation = (f.col('customer_email').rlike(email_regex))
phone_validation = (f.length(f.col('customer_phone'))== 10)
# load_validation = (f.col('load_dt') <= f.current_date())

c_incremental = c_incremental.withColumn('data_quality',f.when(
    (email_validation) & (phone_validation) ,'pass'
).otherwise('fail'))

# Updated records
updated_records = c_incremental.alias('c').join(b_customers.alias('b'),on='customer_id',how='inner').where(
(f.col('c.customer_name') != f.col('b.customer_name')) |
(f.col('c.customer_email') != f.col('b.customer_email')) |
(f.col('c.customer_city') != f.col('b.customer_city')) |
(f.col('c.segment') != f.col('b.segment')) |
(f.col('c.customer_phone') != f.col('b.customer_phone')) 
).select('c.*')

updated_records = updated_records.withColumn('customer_created_dt',f.lit(None)).withColumn('customer_updated_dt',f.current_date()).withColumnRenamed('load_dt','customer_load_dt').select('customer_id','customer_name',     'customer_email','customer_city','segment','customer_phone','customer_load_dt','customer_created_dt','customer_updated_dt','data_quality')

# updated_records.show()

# Inserted records
inserted_records = c_incremental.alias('c').join(b_customers.alias('b'),on='customer_id',how='left_anti').select('c.*')

inserted_records = inserted_records.withColumn('customer_created_dt',f.lit(None)).withColumn('customer_updated_dt',f.current_date()).withColumnRenamed('load_dt','customer_load_dt').select('customer_id','customer_name',     'customer_email','customer_city','segment','customer_phone','customer_load_dt','customer_created_dt','customer_updated_dt','data_quality')

# inserted_records.show()

# silver_customers.show()

# Remove old records before update
silver_customers = (silver_customers.alias('s').join(updated_records.select('customer_id').alias('u'),on='customer_id',how='left_anti'))

silver_customers.unionByName(updated_records).unionByName(inserted_records)

# Orders Incremental 
o_incremental = spark.read.format('csv').option('header','true').schema(o_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoG_V2/orders_incremental.csv')

# Remove Null records
o_incremental = o_incremental.dropna(subset=('order_id'))

# Drop duplicates 
o_incremental = o_incremental.dropDuplicates()

# Fill na
o_incremental = o_incremental.fillna({
    'order_id':0,
    'customer_id':0,
    'product_id':0,
    'amount':0
})

# Filter Invalid records
o_incremental = o_incremental.filter((f.col('amount')>=0))
o_incremental = o_incremental.withColumnRenamed('amount','orders_amount').withColumn('orders_create_dt',f.lit(None)).withColumn('orders_updated_dt',f.lit(f.current_date()))

updated_orders = b_orders.alias('b').join(o_incremental.alias('a'),on='customer_id',how='inner').where(
    (f.col('b.product_id') != f.col('a.product_id')) | (f.col('b.order_id') != f.col('a.order_id')) |
    ((f.col('b.orders_amount') != f.col('a.orders_amount')))
).select('a.*')

new_orders = o_incremental.alias('a').join(b_orders.alias('b'),on='customer_id',how='left_anti').select('a.*')

# updated_orders.show()
# new_orders.show()
# o_incremental.show()

# Remove old records before update
b_orders = (b_orders.alias('b').join(updated_orders.alias('u'),on='customer_id',how='left_anti'))

silver_orders = b_orders.unionByName(updated_orders).unionByName(new_orders)


# Product table 
p_incremental = spark.read.format('csv').option('header','true').schema(p_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoStoG_V2/products_incremental.csv')

p_incremental = p_incremental.dropna(subset=('product_id'))
p_incremental = p_incremental.dropDuplicates()
p_incremental = p_incremental.fillna({
    'product_name':'Unknown',
    'category':'Unknown',
    'brand':'Unknown',
    'price':0,
    'active_flag':'Unknown'
})

p_incremental = p_incremental.withColumn('product_name',f.initcap(f.trim('product_name'))).withColumn('category',f.initcap(f.trim('category'))).withColumn('brand',f.initcap(f.trim('brand'))).withColumn('active_flag',f.initcap(f.trim('active_flag')))

p_incremental = p_incremental.filter(
    (f.col('active_flag') == 'Y') & (f.col('price')>=0)
)

p_incremental = p_incremental.withColumn('products_create_dt',f.lit(None)).withColumn('products_update_dt',f.lit(f.current_date()))

updated_p = b_products.alias('b').join(p_incremental.alias('a'),on='product_id',how='inner').where(
    (f.col('b.product_name') != f.col('a.product_name')) & (f.col('b.category') != f.col('a.category')) & 
    (f.col('b.brand') != f.col('a.brand'))
).select('a.*')

incremental_p = p_incremental.alias('a').join(b_products.alias('b'),on='product_id',how='left_anti').select('a.*')

# Remove the old product data before update
b_products = b_products.alias('p').join(updated_p.alias('u'),on='product_id',how='left_anti')

silver_products = b_products.unionByName(updated_p).unionByName(incremental_p)

gold_orders = silver_orders.alias('o').join(silver_customers.alias('c'),on='customer_id',how='left').join(silver_products.alias('p'),on='product_id',how='left').select('c.customer_id','c.customer_name','c.customer_email','c.customer_city','c.segment','c.customer_phone','p.product_id','p.product_name','p.category','p.brand','p.price','p.active_flag','p.products_create_dt','p.products_update_dt','o.order_id','orders_amount','o.order_dt','orders_create_dt','orders_updated_dt')

gold_orders.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('gold_orders')

# Business rules

# SELECT
#     customer_id AS `Customer ID`,
#     customer_name AS `Customer Name`,
#     customer_city AS `City`,
#     segment AS `Original Segment`,
#     COUNT(order_id) AS `Total Orders`,
#     SUM(orders_amount) AS `Total Sales`,
#     AVG(orders_amount) AS `Average Order Value`,
#     MAX(orders_amount) AS `Highest Order`,
#     COUNT(DISTINCT product_id) AS `Number of Unique Products Purchased`,
#     CASE 
#         WHEN SUM(orders_amount) >= 150000 THEN 'VIP'
#         WHEN SUM(orders_amount) BETWEEN 75000 AND 149999 THEN 'Premium'
#         WHEN SUM(orders_amount) < 75000 THEN 'Regular'
#     END AS customer_category
# FROM gold_orders
# GROUP BY customer_id, customer_name, customer_city, segment;

gold_customer_performance = gold_orders.groupBy('customer_id','customer_name', 'customer_city', 'segment').agg(
    f.count(f.col('order_id')).alias('TotalOrders'),
    f.sum(f.col('orders_amount')).alias('TotalSales'),
    f.round(f.avg(f.col('orders_amount')).alias('AverageOrderValue'),
    f.max(f.col('orders_amount')).alias('HighestOrder'),
    f.countDistinct(f.col('order_id')).alias('NumberofUniqueProductsPurchased')
)
gold_customer_performance = gold_customer_performance.withColumn('Customer_category',
    f.when(f.col('TotalSales') >= 150000, 'VIP')
     .when(f.col('TotalSales') >= 75000, 'Premium')
     .otherwise('Regular')
)

gold_customer_performance.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('gold_customer_performance')


gold_customer_performance.show()